# Custom Memory Extraction for Oracle AI Agent Memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ela689/oracle-ai-developer-hub/blob/feature/custom-memory-extraction-notebook/notebooks/agent_memory/custom_memory_extraction_agent_memory.ipynb) [![Oracle AI Agent Memory docs](https://img.shields.io/badge/Docs-Oracle%20AI%20Agent%20Memory-C74634?logo=oracle&logoColor=white)](https://docs.oracle.com/en/database/oracle/agent-memory/26.6/)

This notebook is an end-to-end customer support use case for domain-specific memory extraction with Oracle AI Agent Memory. It shows how a general extraction baseline can be extended with workflow-specific instructions when an application needs a more precise memory policy.

**Thesis:** memory extraction is a domain policy, not just a summarization step. The agent should preserve useful support facts that can help future work, not every detail from the conversation.

In this support scenario, useful memories include exact order IDs, return requests, tool-confirmed statuses, escalation commitments, and stable delivery preferences. Temporary codes, credentials, payment details, and conversational noise should not become durable memory. The workflow compares baseline extraction with a custom support memory policy, then applies the same pattern to thread-level overrides and tool-result metadata.


## What This Notebook Demonstrates

| Area | What you will run | Why it matters |
|---|---|---|
| FreeSQL connection | Connect to a FreeSQL Oracle Database schema and verify table permissions | The notebook uses a lightweight hosted Oracle Database setup while still exercising real database-backed memory storage |
| Support policy use case | Process a customer-support conversation with order IDs, return requests, delivery issues, escalation commitments, and preferences | The use case gives the extractor clear domain priorities for what should become durable memory |
| Baseline vs custom extraction | Compare general extraction with support-specific extraction instructions | The comparison shows how custom instructions refine memory formation for a specific workflow |
| Thread-level control | Override, update, and clear extraction instructions for one support workflow | Different threads can use different memory policies without changing the global client configuration |
| Tool metadata | Preserve tool-confirmed support facts and inherit shared tool/workflow metadata | Tool tags such as `tool:order_status` make extracted memories easier to govern and retrieve later |
| Scoped retrieval checks | Inspect extracted records and search with metadata filters | Memory should be useful, searchable, and controlled by tenant, source, tag, and workflow |


## Conceptual Flow

The use case starts with one support conversation and sends it through two extraction paths. The baseline path shows the general behavior. The custom path adds a support-specific memory policy that describes durable support facts and exclusions.

| Stage | Baseline path | Custom extraction path |
|---|---|---|
| Input | Same support conversation | Same support conversation plus support policy and metadata |
| Extraction behavior | General extraction baseline | Support-specific policy for durable facts and exclusions |
| Memory output | Baseline memories | Support memories: orders, returns, preferences, escalations, and tool-confirmed facts |
| Storage | Oracle Database records and metadata | Oracle Database records and inherited workflow metadata |
| Later use | General inspection and search | Scoped retrieval for reusable agent memory context |

In short:

```text
Support conversation
        |
        +--> Baseline extraction --> Baseline memories
        |
        +--> Custom support policy + workflow metadata
                 |
                 +--> Support memories: orders, returns, preferences, escalations
                          |
                          +--> Oracle Database records + metadata
                                   |
                                   +--> Scoped retrieval for future agent context
```


# Part 1 - Setup and Database Connection

This section prepares a clean runtime:

1. install the required Python packages;
2. choose an Oracle Database connection, including FreeSQL for a quick hosted schema;
3. configure the LLM and embedding route used by the full example.

FreeSQL is useful when you want a fast Oracle Database connection without running a local container. For a
fully self-contained local run, you can also use Oracle Database Free in Docker.


## Install the Python Packages

Install the package from PyPI. The `[litellm]` extra lets Oracle AI Agent Memory call configured chat and
embedding providers through one interface. `oracledb` is the Oracle Database driver, `pandas` is used to
display results, and `python-dotenv` is a local-development convenience for loading environment values.

Run this once in a clean environment. A dedicated virtual environment is recommended so the database driver,
LLM dependencies, and memory package can be upgraded and tested together.


In [1]:
import sys
from importlib.metadata import version

from IPython.utils.capture import capture_output


assert (3, 10) <= sys.version_info[:2] <= (3, 13), (
    f"Python 3.10-3.13 is required; found {sys.version.split()[0]}"
)

with capture_output():
    get_ipython().run_line_magic(
        "pip",
        "install --upgrade oracleagentmemory[litellm]==26.6.0 oracledb pandas python-dotenv --quiet",
    )

package_version = version("oracleagentmemory")
assert package_version == "26.6.0", (
    f"Oracle AI Agent Memory 26.6.0 is required; found {package_version}"
)

print("Python packages: READY")


Python packages: READY


## Option A - FreeSQL Connection Setup

FreeSQL gives developers a quick Oracle Database schema at [freesql.com](https://freesql.com). After signing in, open **Connect to the Database**, choose **Python**, and copy the username, generated password, and DSN.

Set these values as environment variables before running the notebook:

```text
DB_USER=<FreeSQL schema username>
DB_PASSWORD=<generated FreeSQL schema password>
DB_CONNECT_STRING=tcps://db.freesql.com:2484/<FreeSQL service name>
# DB_DSN is also accepted as an alias for DB_CONNECT_STRING.
```

The next setup cell opens a real Oracle Database connection and verifies that the schema can create and remove a small table.


## Optional Fallback - Standalone Oracle Database Setup with Docker

Use Oracle Database Free in Docker when you want a reproducible local database for the full example.

```bash
docker pull gvenzl/oracle-free:23.26.2
docker volume create oracle-free-data

docker run -d \
  --name oracle-ai-memory-example \
  -p 1522:1521 \
  -e ORACLE_PASSWORD="<admin-password>" \
  -e APP_USER="<app-user>" \
  -e APP_USER_PASSWORD="<app-password>" \
  -v oracle-free-data:/opt/oracle/oradata \
  gvenzl/oracle-free:23.26.2

docker logs -f oracle-ai-memory-example
```

Then configure the active connection:

```text
DB_USER=<app-user>
DB_PASSWORD=<app-password>
DB_CONNECT_STRING=localhost:1522/FREEPDB1
```


## Runtime Configuration

The notebook needs one Oracle Database connection and one model provider key for memory extraction:

| Variable | Purpose |
|---|---|
| `DB_USER` | Oracle Database schema user |
| `DB_PASSWORD` | Password for the schema user |
| `DB_CONNECT_STRING` | Oracle Database connect string or descriptor |
| `MODEL_PROVIDER_API_KEY` | Provider key used by the configured memory extraction LLM |
| `MEMORY_LLM_MODEL` | Chat model used for memory extraction |

For retrieval, use one of these routes:

| Route | When to use it | Required variables |
|---|---|---|
| `EMBED_BACKEND=provider` | Default FreeSQL-friendly route for this custom extraction notebook | none beyond the model provider key |
| `EMBED_BACKEND=indb` | Oracle-native route when a database-resident embedding model is available | `DB_EMBED_MODEL`, `DB_EMBED_DIM` |

The default route keeps retrieval lightweight so the notebook can focus on custom extraction behavior. In an Oracle AI Database environment with a visible embedding model, switch to `EMBED_BACKEND=indb` to use `OracleDBEmbedder`.


## Load Local Environment Values

This cell loads local environment values when present. Public runners can set the same values directly as
environment variables.


In [2]:
from pathlib import Path

from dotenv import load_dotenv


load_dotenv(Path.cwd() / ".env")

print("Runtime configuration loader: READY")


Runtime configuration loader: READY


## Read and Validate Configuration

The notebook validates only whether required values exist. It does not print usernames, passwords, DSNs,
schema names, database versions, or local paths.


In [3]:
import os

import pandas as pd


CONFIG = {
    "DB_USER": os.getenv("DB_USER") or os.getenv("ORACLE_USER"),
    "DB_PASSWORD": os.getenv("DB_PASSWORD") or os.getenv("ORACLE_PASSWORD"),
    "DB_CONNECT_STRING": os.getenv("DB_CONNECT_STRING") or os.getenv("DB_DSN") or os.getenv("ORACLE_DSN"),
    "MODEL_PROVIDER_API_KEY": os.getenv("MODEL_PROVIDER_API_KEY") or os.getenv("MEMORY_LLM_API_KEY"),
    "MEMORY_LLM_API_BASE": os.getenv("MEMORY_LLM_API_BASE"),
    "MEMORY_LLM_API_TYPE": os.getenv("MEMORY_LLM_API_TYPE", "chat_completions"),
    "MEMORY_LLM_MODEL": os.getenv("MEMORY_LLM_MODEL") or "gpt-4o-mini",
    "EMBED_BACKEND": os.getenv("EMBED_BACKEND", "provider").lower(),
    "EMBED_MODEL": os.getenv("EMBED_MODEL", "text-embedding-3-small"),
    "EMBED_DIM": int(os.getenv("EMBED_DIM", "1536")),
    "DB_EMBED_MODEL": os.getenv("DB_EMBED_MODEL") or os.getenv("ORACLE_DB_EMBEDDING_MODEL"),
    "DB_EMBED_DIM": os.getenv("DB_EMBED_DIM") or os.getenv("ORACLE_DB_EMBEDDING_DIMENSION"),
    "DB_EMBED_INPUT": os.getenv("DB_EMBED_INPUT", "DATA"),
}


required = {
    "DB_USER": "DB_USER or ORACLE_USER",
    "DB_PASSWORD": "DB_PASSWORD or ORACLE_PASSWORD",
    "DB_CONNECT_STRING": "DB_CONNECT_STRING, DB_DSN, or ORACLE_DSN",
    "MODEL_PROVIDER_API_KEY": "MODEL_PROVIDER_API_KEY or MEMORY_LLM_API_KEY",
}
missing = [label for key, label in required.items() if not CONFIG.get(key)]
if missing:
    raise RuntimeError(
        "Missing required environment values: "
        + ", ".join(missing)
        + ". Add them before running the notebook."
    )

print("Required configuration: READY")


Required configuration: READY


## Create and Verify the Oracle Database Connection Pool

This cell creates the Oracle Database connection pool used by the notebook. When the connection string points to FreeSQL, it also verifies the FreeSQL setup requested for this content.

A pool is used instead of a single raw connection because Oracle AI Agent Memory can perform several kinds of database work during a realistic application flow: message writes, memory extraction, memory inspection, and retrieval. Even in a notebook, using a pool mirrors the application pattern more closely.

The check intentionally prints only readiness status. It does not display usernames, schema names, DSNs, passwords, database versions, or local paths.

It verifies:

- Python can connect to the configured Oracle Database service;
- a simple `SELECT 1 FROM dual` query succeeds;
- the schema can create, insert into, read from, and drop a small table.

The table-permission check matters because Oracle AI Agent Memory creates and manages its own database objects during schema setup.

The connection also sets a Developer Hub program identifier before the pool is created, so usage can be recognized in Oracle Developer Hub telemetry.


In [4]:
import oracledb
from uuid import uuid4


oracledb.defaults.program = "devrel-developerhub-custom-memory-extraction-agent-memory"


db_pool = oracledb.create_pool(
    user=CONFIG["DB_USER"],
    password=CONFIG["DB_PASSWORD"],
    dsn=CONFIG["DB_CONNECT_STRING"],
    min=1,
    max=4,
    increment=1,
)

permission_check_table = f"OAM_FREESQL_CHECK_{uuid4().hex[:8].upper()}"

try:
    with db_pool.acquire() as connection:
        with connection.cursor() as cursor:
            cursor.execute("SELECT 1 FROM dual").fetchone()
            cursor.execute(
                f"""
                CREATE TABLE {permission_check_table} (
                    id NUMBER PRIMARY KEY,
                    note VARCHAR2(100)
                )
                """
            )
            cursor.execute(
                f"INSERT INTO {permission_check_table} (id, note) VALUES (:id, :note)",
                id=1,
                note="table permission check",
            )
            count = cursor.execute(
                f"SELECT COUNT(*) FROM {permission_check_table}"
            ).fetchone()[0]
            if count != 1:
                raise RuntimeError("Table permission check returned an unexpected row count.")
        connection.commit()
finally:
    try:
        with db_pool.acquire() as cleanup_connection:
            with cleanup_connection.cursor() as cursor:
                cursor.execute(f"DROP TABLE {permission_check_table} PURGE")
            cleanup_connection.commit()
    except oracledb.Error:
        pass

print("FreeSQL connection and table permissions: READY")


FreeSQL connection and table permissions: READY


## Choose the Retrieval Route

This notebook focuses on custom memory extraction, so the default FreeSQL path uses keyword retrieval. That keeps the workflow lightweight while still using Oracle AI Agent Memory for database-backed storage, extraction, thread control, metadata, and scoped search.

For Oracle AI Database environments where a database-resident embedding model is available, switch to `EMBED_BACKEND=indb` to use `OracleDBEmbedder` and hybrid retrieval with the same memory APIs.


In [5]:
from oracleagentmemory.core import SearchIndexSyncMode, SearchStrategy
from oracleagentmemory.core.embedders import Embedder, OracleDBEmbedder


EMBED_BACKEND = CONFIG["EMBED_BACKEND"]

if EMBED_BACKEND == "indb":
    missing_indb = [
        name
        for name in ["DB_EMBED_MODEL", "DB_EMBED_DIM"]
        if not CONFIG.get(name) or str(CONFIG.get(name)).startswith("<")
    ]
    if missing_indb:
        raise RuntimeError(
            "EMBED_BACKEND=indb requires: "
            + ", ".join(missing_indb)
            + ". Use EMBED_BACKEND=provider for a portable run, or configure a visible database embedding model."
        )

    embedder = OracleDBEmbedder(
        connection=db_pool,
        model=CONFIG["DB_EMBED_MODEL"],
        input_name=CONFIG["DB_EMBED_INPUT"],
        embedding_dimension=int(CONFIG["DB_EMBED_DIM"]),
        max_input_tokens=512,
        normalize=True,
        batch_size=16,
    )
    SEARCH_STRATEGY = SearchStrategy.HYBRID
    SEARCH_INDEX_SYNC = SearchIndexSyncMode.ON_COMMIT
    VECTOR_DIM = None
    print("Embedding route: OracleDBEmbedder")

elif EMBED_BACKEND == "provider":
    # The portable FreeSQL path keeps retrieval keyword-based so the notebook
    # can run without creating database vector indexes. The LLM provider is
    # still used for memory extraction.
    embedder = None
    SEARCH_STRATEGY = SearchStrategy.KEYWORD
    SEARCH_INDEX_SYNC = None
    VECTOR_DIM = None
    print("Retrieval route: portable keyword search")

else:
    raise RuntimeError("EMBED_BACKEND must be either 'indb' or 'provider'.")


Retrieval route: portable keyword search


## Configure the Memory Extraction LLM

The LLM is used by Oracle AI Agent Memory to transform raw conversation messages into durable memory records. Retrieval configuration is handled separately, so the notebook can demonstrate the extraction policy independently from the retrieval backend.


In [6]:
from oracleagentmemory.core.llms import Llm, LlmApiType


llm_kwargs = {
    "model": CONFIG["MEMORY_LLM_MODEL"],
    "api_key": CONFIG["MODEL_PROVIDER_API_KEY"],
    "temperature": 0,
    "max_tokens": 2_000,
}
if CONFIG["MEMORY_LLM_API_BASE"]:
    llm_kwargs["api_base"] = CONFIG["MEMORY_LLM_API_BASE"]
if CONFIG["MEMORY_LLM_API_TYPE"] == "responses":
    llm_kwargs["api_type"] = LlmApiType.RESPONSES

memory_llm = Llm(**llm_kwargs)

print("Memory extraction LLM: READY")


Memory extraction LLM: READY


# Part 2 - Extraction Policy and Memory Clients

This section defines the support-specific extraction policy and creates two memory clients over the same database-backed store:

- a baseline client that uses the package's general extraction behavior;
- a custom client that adds support-specific extraction instructions.


## Define Domain-Specific Extraction Instructions

Custom extraction instructions are appended to the built-in extraction prompt. In this example, the instructions define the support team's memory policy: preserve durable business facts and exact identifiers, and avoid temporary or sensitive details.

Good instructions describe business intent and exclusions. They should guide memory formation without depending on provider-specific output formatting.


In [7]:
SUPPORT_EXTRACTION_INSTRUCTIONS = """
Extract only durable customer-support memory that can help future support interactions.

Preserve:
- confirmed order identifiers, return identifiers, case identifiers, and return requests;
- delivery problems, product defects, replacement commitments, and escalation reasons;
- stable customer preferences that should influence future interactions;
- tool-derived support facts when the source message metadata identifies a tool result.

Ignore:
- greetings, small talk, apologies, and temporary conversational wording;
- speculation or unconfirmed guesses;
- credentials, payment secrets, one-time verification codes, and unnecessary sensitive data.

Prefer concise facts, preferences, or guidelines. Keep exact identifiers when they matter.
""".strip()

print("Support extraction policy: READY")


Support extraction policy: READY


## Define Example Scope

The run-specific IDs keep repeated notebook executions isolated from each other.


In [8]:
from uuid import uuid4


RUN_ID = uuid4().hex[:8]
MEMORY_STORE_ID = "CUST_EXTRACT"
USER_ID = f"customer_custom_extract_{RUN_ID}"
AGENT_ID = f"support_agent_{RUN_ID}"

print("Example scope: READY")


Example scope: READY


## Create the Database Stores

The helper below passes `search_index_sync` only when the selected search strategy supports it. This matters
because vector-only search does not use the text-search index sync setting.


In [9]:
import contextlib
import io
import warnings

from oracleagentmemory.core import OracleDBMemoryStore, SchemaPolicy


def build_store(schema_policy):
    store_kwargs = {
        "pool": db_pool,
        "embedder": embedder,
        "memory_store_id": MEMORY_STORE_ID,
        "schema_policy": schema_policy,
        "search_strategy": SEARCH_STRATEGY,
        "vector_dim": VECTOR_DIM,
    }
    if SEARCH_INDEX_SYNC is not None:
        store_kwargs["search_index_sync"] = SEARCH_INDEX_SYNC
    return OracleDBMemoryStore(**store_kwargs)


suppressed_stderr = io.StringIO()
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=UserWarning)
    with contextlib.redirect_stderr(suppressed_stderr):
        base_store = build_store(SchemaPolicy.CREATE_IF_NECESSARY)
        custom_store = build_store(SchemaPolicy.REQUIRE_EXISTING)

print("Database-backed memory stores: READY")


Database-backed memory stores: READY


## Create Baseline and Custom Memory Clients

The baseline client uses the general extraction behavior. The custom client uses the same database store and LLM, but adds support-specific extraction instructions through `MemoryExtractionConfig`.


In [10]:
from oracleagentmemory.core import MemoryExtractionConfig, OracleAgentMemory


base_memory = OracleAgentMemory(
    store=base_store,
    llm=memory_llm,
    memory_extraction_config=MemoryExtractionConfig(
        memory_extraction_frequency=1,
        enable_context_summary=False,
    ),
)

custom_memory = OracleAgentMemory(
    store=custom_store,
    llm=memory_llm,
    memory_extraction_config=MemoryExtractionConfig(
        memory_extraction_frequency=1,
        enable_context_summary=False,
        memory_extraction_custom_instructions=SUPPORT_EXTRACTION_INSTRUCTIONS,
    ),
)

print("Default and custom memory clients: READY")


Default and custom memory clients: READY


# Part 3 - Baseline vs Custom Extraction

This is the core use-case section. The same support conversation is processed twice so the difference is easy to inspect:

- **Baseline extraction** shows the general extraction behavior.
- **Custom extraction** adds a support memory policy for durable support facts, exact identifiers, and temporary or sensitive details that should be ignored.


## Prepare the Support Conversation

The conversation deliberately includes information with different memory lifecycles.

Durable support facts:

- order ID and damaged product detail;
- return request and replacement delivery intent;
- stable delivery preference;
- escalation condition.

Short-lived or sensitive details:

- greeting and conversational wording;
- temporary lobby code;
- anything that looks like a one-time or sensitive detail.

This makes the use case practical: the custom policy is not trying to replace general extraction; it adds domain guidance for what this support workflow considers reusable memory.


In [11]:
SUPPORT_CONVERSATION = [
    {
        "role": "user",
        "content": (
            "Hi! Order ORD-7421 arrived damaged. The left hinge is cracked, "
            "and I need a return request plus a replacement delivery. "
            "My temporary lobby code today is 4812, but please do not save that."
        ),
    },
    {
        "role": "assistant",
        "content": (
            "I can help with order ORD-7421. I will start a return request and "
            "mark the damaged hinge as the replacement reason."
        ),
    },
    {
        "role": "user",
        "content": (
            "For future deliveries, I prefer morning delivery windows. "
            "Please escalate if the replacement cannot ship this week."
        ),
    },
]

print("Support conversation: READY")


Support conversation: READY


## Run Baseline Extraction

Baseline extraction provides a general-purpose comparison point. It can capture useful facts from the conversation, but it does not apply the support-specific policy defined above. This lets us compare the general behavior with a custom policy for durable support facts, exact identifiers, and temporary details that should be ignored.


In [12]:
default_thread = base_memory.create_thread(
    thread_id=f"default_support_{RUN_ID}",
    user_id=USER_ID,
    agent_id=AGENT_ID,
)

await default_thread.add_messages_async(SUPPORT_CONVERSATION)
await default_thread.wait_for_memory_extraction_async()

print("Default extraction thread: READY")


Default extraction thread: READY


## Run Custom Support Extraction

The same conversation is now processed with custom extraction instructions. These instructions add the support workflow policy: preserve durable facts such as order IDs, return requests, delivery preferences, and escalation commitments, while avoiding temporary or sensitive details.


In [13]:
custom_thread = custom_memory.create_thread(
    thread_id=f"custom_support_{RUN_ID}",
    user_id=USER_ID,
    agent_id=AGENT_ID,
)

await custom_thread.add_messages_async(SUPPORT_CONVERSATION)
await custom_thread.wait_for_memory_extraction_async()

print("Custom extraction thread: READY")


Custom extraction thread: READY


## Compare Extracted Memories

First inspect the records created by the two extraction paths. This is the central comparison in the notebook:

- **Baseline extraction** shows general memory behavior.
- **Custom extraction** shows support-specific memory behavior shaped by the extraction policy.

The comparison should make the before/after difference easy to see. Look for whether the custom path preserves durable support facts more directly, such as order IDs, return or replacement details, delivery preferences, escalation commitments, and tool-confirmed support state.

The other side of the comparison is equally important: temporary details, such as the lobby code, should not become durable memory.


In [14]:
RECORD_TYPES_TO_INSPECT = ["memory", "fact", "preference", "guideline"]


def record_to_row(record):
    return {
        "record_type": record.record_type,
        "content": getattr(record, "content", None) or getattr(record, "text", None),
        "metadata": record.metadata,
    }


def list_extracted_records(store, thread_id):
    rows = []
    for record_type in RECORD_TYPES_TO_INSPECT:
        for record in store.list(record_type, thread_id=thread_id, limit=None):
            rows.append(record_to_row(record))
    return rows


async def search_thread_results(thread, queries, *, max_results=5):
    seen = {}
    for query in queries:
        for result in await thread.search_async(
            query,
            max_results=max_results,
            exact_thread_match=True,
            record_types=RECORD_TYPES_TO_INSPECT,
        ):
            record = result.record
            key = getattr(record, "id", None) or (record.record_type, result.content)
            seen[key] = {
                "record_type": record.record_type,
                "content": result.content,
                "metadata": record.metadata,
            }
    return list(seen.values())


print("Memory inspection helpers: READY")


Memory inspection helpers: READY


### Inspect the Stored Memory Records

The table below compares the records created by default extraction and by the custom support policy.


In [15]:
default_memories = list_extracted_records(base_store, default_thread.thread_id)
custom_memories = list_extracted_records(custom_store, custom_thread.thread_id)

comparison_columns = ["extraction", "record_type", "content", "metadata"]
comparison = pd.DataFrame(
    [{"extraction": "Default", **row} for row in default_memories]
    + [{"extraction": "Custom support policy", **row} for row in custom_memories],
    columns=comparison_columns,
)

if comparison.empty:
    pd.DataFrame([{"status": "No extracted memory records found. Check the extraction LLM configuration and rerun the extraction cells."}])
else:
    comparison[comparison_columns]


### Lightweight Quality Check

This check is intentionally simple and readable. It does not grade the model or claim benchmark quality. It verifies the expected behavior for this support use case.

Expected result:

- the custom memory output keeps the order ID;
- it keeps return or replacement details;
- it keeps the stable delivery preference;
- it avoids saving temporary details such as the lobby code.

These checks make the notebook easier to review because they turn the policy goal into concrete output expectations.


In [16]:
def score_memory_set(rows):
    text = " ".join(str(row["content"]).lower() for row in rows)
    return {
        "mentions_order_id": "ord-7421" in text or "7421" in text,
        "mentions_return_or_replacement": any(
            token in text for token in ["return", "replacement", "replace"]
        ),
        "mentions_delivery_preference": "morning" in text and "delivery" in text,
        "temporary_code_leaked": "4812" in text,
    }


quality = pd.DataFrame(
    [
        {"extraction": "Default", **score_memory_set(default_memories)},
        {"extraction": "Custom support policy", **score_memory_set(custom_memories)},
    ]
)

quality


,extraction,mentions_order_id,mentions_return_or_replacement,mentions_delivery_preference,temporary_code_leaked
0,Default,True,True,True,False
1,Custom support policy,True,True,True,False


# Part 4 - Thread-Level Control and Tool Metadata


## Override Instructions for One Thread

Client-level instructions are the default. A thread-level override is useful when one support conversation has
a narrower workflow, such as escalation handling, replacement tracking, or incident follow-up.


In [17]:
ESCALATION_EXTRACTION_INSTRUCTIONS = """
Extract only durable escalation-support facts:
- escalation reasons;
- replacement shipment identifiers;
- customer-facing follow-up commitments;
- deadlines or status checks that should influence future support turns.
Ignore general delivery preferences and unrelated return-request details.
""".strip()

print("Escalation extraction policy: READY")


Escalation extraction policy: READY


### Run an Escalation-Specific Thread

This thread overrides the broader support policy with narrower escalation-specific instructions for one
conversation.


In [18]:
escalation_thread = custom_memory.create_thread(
    thread_id=f"escalation_support_{RUN_ID}",
    user_id=USER_ID,
    agent_id=AGENT_ID,
    memory_extraction_config=MemoryExtractionConfig(
        memory_extraction_frequency=1,
        enable_context_summary=False,
        memory_extraction_custom_instructions=ESCALATION_EXTRACTION_INSTRUCTIONS,
    ),
)

await escalation_thread.add_messages_async(
    [
        {
            "role": "user",
            "content": (
                "Replacement shipment RMA-8842 for order ORD-7421 still has not shipped. "
                "Please escalate because the customer needs a status update by Friday."
            ),
        },
        {
            "role": "assistant",
            "content": (
                "I will escalate replacement shipment RMA-8842 and track the Friday follow-up commitment."
            ),
        },
    ]
)
await escalation_thread.wait_for_memory_extraction_async()

print("Escalation extraction thread: READY")


Escalation extraction thread: READY


### Inspect Escalation Memories

Inspect the records extracted from the overridden thread. The expected output is narrower than the broader support policy because this thread focuses on escalation and shipment follow-up.


In [19]:
escalation_results = list_extracted_records(custom_store, escalation_thread.thread_id)

pd.DataFrame(
    escalation_results,
    columns=["record_type", "content", "metadata"],
)[["record_type", "content", "metadata"]]


,record_type,content,metadata
0,fact,Replacement shipment RMA-8842 for order ORD-74...,{'$agent_memory': {'extraction_id': 'escalatio...
1,fact,A follow-up commitment to provide a status upd...,{'$agent_memory': {'extraction_id': 'follow_up...


## Update and Clear Thread-Level Instructions

`update_thread()` persists runtime configuration changes. Passing a partial
`MemoryExtractionConfig` updates only the provided fields. Passing
`memory_extraction_custom_instructions=None` clears the stored override.


In [20]:
updated_escalation_thread = custom_memory.update_thread(
    escalation_thread.thread_id,
    memory_extraction_config=MemoryExtractionConfig(
        memory_extraction_custom_instructions=(
            "Extract only shipment escalation commitments and customer-facing follow-up deadlines."
        )
    ),
)

cleared_escalation_thread = custom_memory.update_thread(
    escalation_thread.thread_id,
    memory_extraction_config=MemoryExtractionConfig(
        memory_extraction_custom_instructions=None
    ),
)

print("Thread-level extraction instructions updated and cleared: READY")


Thread-level extraction instructions updated and cleared: READY


## Add Tool-Result Metadata to a Support Thread

Richeek's feedback also called out tool usage extraction as a useful custom extraction scenario. In a real support agent, reliable operational facts often come from tools: order status, shipment status, return authorization, entitlement checks, or case-management systems.

This section represents a tool result as an assistant message and attaches shared workflow metadata to the message batch. The custom extraction policy preserves the tool-confirmed business fact, while metadata inheritance adds tags such as `tool:order_status` to the extracted memory.

That gives the application two useful controls later:

- the memory text captures the durable support fact;
- the metadata records where the fact came from and how it should be scoped.


In [21]:
TOOL_AWARE_EXTRACTION_INSTRUCTIONS = """
Extract durable support facts from user, assistant, and tool-result messages.
When the conversation contains a tool-confirmed status, preserve the business fact
but do not copy raw payload wording unless it is needed for follow-up.
Keep exact order IDs, shipment status, escalation reason, and customer-facing commitment.
""".strip()

tool_metadata = {
    "tenant": "acme",
    "source": "support-copilot",
    "tags": ["support", "tool:order_status", "replacement"],
}

print("Tool-aware policy and metadata: READY")


Tool-aware policy and metadata: READY


### Run a Tool-Aware Thread

The metadata is shared across the message batch so the selected metadata keys can be inherited safely by
extracted memories.


In [22]:
tool_thread = custom_memory.create_thread(
    thread_id=f"tool_support_{RUN_ID}",
    user_id=USER_ID,
    agent_id=AGENT_ID,
    memory_extraction_config=MemoryExtractionConfig(
        memory_extraction_frequency=1,
        enable_context_summary=False,
        memory_extraction_custom_instructions=TOOL_AWARE_EXTRACTION_INSTRUCTIONS,
        memory_extraction_inherit_message_metadata=["tenant", "source", "tags"],
    ),
)

await tool_thread.add_messages_async(
    [
        {
            "role": "user",
            "content": "Can you check the replacement status for order ORD-7421?",
        },
        {
            "role": "assistant",
            "content": (
                "Tool result from order_status: replacement shipment RMA-8842 for order "
                "ORD-7421 is delayed because the hinge part is backordered. Escalate if "
                "it does not ship by Friday."
            ),
        },
    ],
    metadata=tool_metadata,
)
await tool_thread.wait_for_memory_extraction_async()

print("Tool-aware extraction thread: READY")


Tool-aware extraction thread: READY


## Inspect Tool-Aware Memories

Before applying metadata filters, inspect the memories extracted from the tool-aware support thread.


In [23]:
tool_stored_rows = list_extracted_records(custom_store, tool_thread.thread_id)

pd.DataFrame(
    tool_stored_rows,
    columns=["record_type", "content", "metadata"],
)[["record_type", "content", "metadata"]]


,record_type,content,metadata
0,fact,Replacement shipment RMA-8842 for order ORD-74...,"{'tenant': 'acme', 'source': 'support-copilot'..."
1,guideline,Escalate the order status if the replacement s...,"{'tenant': 'acme', 'source': 'support-copilot'..."


## Search Extracted Memories by Tool Metadata

After tool-aware extraction, metadata filters let the application retrieve memories that belong to a selected support workflow. This is the governance side of custom extraction: the memory can be useful while still being scoped by tenant, source, and tool tag.


In [24]:
tool_filtered_results = await tool_thread.search_async(
    "replacement shipment delayed backordered hinge escalate",
    max_results=5,
    exact_thread_match=True,
    record_types=["memory", "fact", "preference", "guideline"],
    metadata_filter={
        "tenant": "acme",
        "source": "support-copilot",
        "tags": ["support", "tool:order_status", "replacement"],
    },
)

tool_filtered_rows = [
    {
        "record_type": result.record.record_type,
        "content": result.content,
        "metadata": result.record.metadata,
    }
    for result in tool_filtered_results
]

if tool_filtered_rows:
    pd.DataFrame(tool_filtered_rows)
else:
    pd.DataFrame(
        [{"status": "No matching memories found. Re-run the tool-aware extraction cell before searching."}]
    )


### Array Membership Filter Variant

When metadata values are arrays, exact list matching is usually too strict. Array operators
such as `$array_contains`, `$array_contains_any`, and `$not` express workflow filters more
directly.


In [25]:
tool_membership_results = await tool_thread.search_async(
    "order replacement status",
    max_results=5,
    exact_thread_match=True,
    record_types=["memory", "fact", "preference", "guideline"],
    metadata_filter={
        "tenant": "acme",
        "tags": {
            "$array_contains": "tool:order_status",
            "$array_contains_any": ["replacement", "support"],
            "$not": {"$array_contains": "returns"},
        },
    },
)

membership_rows = [
    {
        "record_type": result.record.record_type,
        "content": result.content,
        "metadata": result.record.metadata,
    }
    for result in tool_membership_results
]

if membership_rows:
    pd.DataFrame(membership_rows)
else:
    pd.DataFrame(
        [{"status": "No matching memories found for the array membership filter."}]
    )


# Part 5 - Production Notes and Cleanup

The workflow is intentionally small, but the same patterns map to production. In enterprise agents, memory extraction should reflect workflow priorities, privacy boundaries, and domain-specific facts instead of summarizing everything the user said.

This matters because better memory policy can lead to:

- fewer noisy memories;
- more consistent agent behavior across sessions;
- better retrieval because durable facts are easier to search and scope;
- stronger compliance posture because sensitive or temporary details are less likely to become durable memory;
- clearer ownership of what the application considers useful memory.

Operationally, production applications should also:

- keep authentication and authorization in the surrounding application;
- scope every memory operation by user, agent, thread, and metadata;
- treat custom extraction as policy guidance, not a hard redaction guarantee;
- avoid sending secrets or unnecessary sensitive content into memory ingestion;
- monitor extraction failures and queue pressure when using background extraction;
- use TTL and delete APIs for lifecycle control;
- configure scheduler-job privileges when administrators want expired records to be physically purged automatically.


## Cleanup and Shutdown

The cleanup step removes the run-scoped example user and then closes application-owned resources.

The order matters:

1. delete the example memory records created by this run;
2. close the Oracle AI Agent Memory clients so no more memory operations use the pool;
3. close the Oracle Database pool last.

Keeping this at the end makes repeated notebook runs easier and avoids leaving unnecessary example data in shared development schemas.


In [26]:
cleanup_notes = []

if "custom_memory" in globals():
    try:
        custom_memory.delete_user(USER_ID, cascade=True)
        cleanup_notes.append("example records removed")
    except Exception:
        cleanup_notes.append("example record cleanup skipped")

for client_name in ["base_memory", "custom_memory"]:
    client = globals().get(client_name)
    if client is not None:
        try:
            client.close()
        except Exception:
            pass

if "db_pool" in globals():
    try:
        db_pool.close(force=True)
        cleanup_notes.append("database pool closed")
    except Exception:
        cleanup_notes.append("database pool close skipped")

print("Cleanup and shutdown: READY")


Cleanup and shutdown: READY


## What This Notebook Shows

This notebook demonstrates how Oracle AI Agent Memory can make extraction policy explicit for a concrete customer support use case:

1. **FreeSQL-ready setup:** the notebook connects to a FreeSQL Oracle Database schema and verifies basic table permissions before using Oracle AI Agent Memory.
2. **Domain-specific extraction:** custom instructions guide memory formation for order IDs, return requests, replacement context, delivery preferences, escalation commitments, and tool-confirmed facts.
3. **Baseline vs custom behavior:** the same conversation is processed through both paths so the stored memories can be compared directly.
4. **Thread-level control:** one support workflow can override, update, or clear extraction behavior without changing the global client policy.
5. **Tool-aware memory:** tool-result metadata can flow into memories and support scoped retrieval later.
6. **Governed retrieval:** metadata filters complement relevance so applications can constrain memory by tenant, source, tag, and workflow.

For production, keep authentication and authorization in the surrounding application, avoid storing secrets, size database pools for concurrent extraction and retrieval, and use TTL/delete APIs for lifecycle control.

## Resources

- [Oracle AI Agent Memory package](https://pypi.org/project/oracleagentmemory/26.6.0/)
- [Oracle AI Agent Memory documentation](https://docs.oracle.com/en/database/oracle/agent-memory/26.4/)
- [FreeSQL](https://freesql.com/)
- [Oracle AI Vector Search documentation](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/overview-ai-vector-search.html)
